## GPU check and installs

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU found. Runtime > Change runtime type > T4 GPU, then run this cell again."
print(gpu.stdout.strip())

In [ ]:
# Squad 1
!pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q -U paddleocr "nvidia-nccl-cu12>=2.27.3"
!apt-get update -qq          
!apt-get install -y -qq poppler-utils
!pip install -q pdf2image opencv-python-headless supabase "httpx==0.28.1" "pydantic==2.14.1"
# Squad 2 and Squad 3
!pip install -q Pillow google-genai pymupdf
!python -c "import paddle; paddle.utils.run_check()"
!pdftoppm -v                 

## Get the project and connect to Supabase

In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path

REPO_URL = ""          # empty = upload the zip instead
WORK = Path("/content/project")

if not WORK.exists():
    WORK.mkdir(parents=True)
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(WORK)], check=True)
    else:
        from google.colab import files
        up = files.upload()                                  
        zname = next(iter(up))
        with zipfile.ZipFile(zname) as z:
            z.extractall(WORK)

ROOT = next(p.parent for p in WORK.rglob("OCR_reader") if (p.parent / "orchestrator").exists())
sys.path.insert(0, str(ROOT / "orchestrator"))
print("Project root:", ROOT)

In [ ]:
from google.colab import userdata
for key in ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "GEMINI_API_KEY"):      
    os.environ[key] = userdata.get(key)

from supabase import create_client
sb = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_KEY"])

sb.table("claims_cases").select("case_id").limit(1).execute()
sb.table("claim_documents").select("doc_id").limit(1).execute()
for b in ("raw-medical-docs", "processed-previews"):
    sb.storage.from_(b).list(path="", options={"limit": 1})
print("Supabase connected: tables and buckets OK")

## the pipeline

In [ ]:
import pipeline, intake, human_review, review_ui

ns = pipeline.load_squad1(ROOT, sb)        
P = pipeline.Pipeline(sb, ROOT, squad1_ns=ns)  
print("Pipeline ready. (The OCR model loads on first use, so the first document takes longer.)")

## Feed a new case

In [ ]:
from google.colab import files
CASE_NUMBER = "PA-2026-0001"                # <- edit: one case number per patient case

inbox = Path("/content/inbox") / CASE_NUMBER
inbox.mkdir(parents=True, exist_ok=True)
for name, data in files.upload().items():
    (inbox / name).write_bytes(data)

summary = intake.intake_case(sb, CASE_NUMBER, [inbox])
print(summary)
P.run_until_settled()

## Human review

In [ ]:
review_ui.show(sb, P)

## Status Check

In [ ]:
pipeline.print_status(sb)

## Answer an RFI

In [ ]:
from google.colab import files
CASE_NUMBER = "PA-2026-0001"                # <- the case waiting for a rescan
inbox = Path("/content/rescan") / CASE_NUMBER
inbox.mkdir(parents=True, exist_ok=True)
for name, data in files.upload().items():
    (inbox / name).write_bytes(data)
print(intake.intake_case(sb, CASE_NUMBER, [inbox], replace=True))
P.run_until_settled()

**Keep watching for new work** (stops after `max_minutes`, or press the stop button):

In [ ]:
P.watch(poll_seconds=30, max_minutes=60)